In [1]:
# ---------------------------------------------
# HEART ATTACK RISK PREDICTION (Smartwatch-Based)
# ---------------------------------------------

import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# ---------------------------------------------
# 1️⃣ Define dataset (simulate smartwatch data)
# ---------------------------------------------
# Columns: same as predict.html form
columns = [
    'age', 'sex', 'blood_oxygen', 'heart_rate',
    'stress_level', 'weight', 'height',
    'smoking', 'drinking'
]

# Simulated example data (you can add more rows)
data = [
    [25, 1, 98, 75, 3, 65, 175, 0, 0],
    [40, 0, 95, 90, 6, 70, 160, 0, 0],
    [55, 1, 92, 110, 8, 82, 170, 1, 1],
    [60, 1, 88, 120, 9, 90, 165, 1, 1],
    [35, 0, 97, 80, 4, 60, 168, 0, 1],
    [50, 1, 90, 105, 7, 75, 172, 1, 0],
    [45, 0, 94, 95, 5, 68, 162, 0, 0],
    [65, 1, 89, 115, 9, 85, 170, 1, 1],
    [30, 0, 99, 70, 2, 55, 160, 0, 0],
    [70, 1, 85, 125, 10, 95, 168, 1, 1]
]

# Target output (0 = Low, 1 = Medium, 2 = High)
# You can adjust this mapping later for better accuracy
y = [0, 0, 2, 2, 1, 1, 0, 2, 0, 2]

# Convert to DataFrame
df = pd.DataFrame(data, columns=columns)
df['bmi'] = df['weight'] / ((df['height']/100) ** 2)  # Add BMI column

# ---------------------------------------------
# 2️⃣ Split data for training
# ---------------------------------------------
X = df[['age', 'sex', 'blood_oxygen', 'heart_rate', 'stress_level',
        'weight', 'height', 'smoking', 'drinking', 'bmi']]
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------------------------------------------
# 3️⃣ Scale and Train Model
# ---------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

print("✅ Model training complete!")

# ---------------------------------------------
# 4️⃣ Save Model & Scaler
# ---------------------------------------------
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("✅ Model and Scaler saved as model.pkl and scaler.pkl")

# ---------------------------------------------
# 5️⃣ Test Predictions
# ---------------------------------------------
# Sample inputs (same as your form fields)
samples = [
    [45, 1, 95, 100, 5, 70, 168, 1, 0],  # Example input
    [30, 0, 98, 75, 2, 60, 160, 0, 0],
    [60, 1, 88, 120, 9, 85, 170, 1, 1]
]

for s in samples:
    height_m = s[6] / 100
    bmi = s[5] / (height_m ** 2)
    s.append(bmi)
    df_sample = pd.DataFrame([s], columns=X.columns)
    scaled = scaler.transform(df_sample)
    prob = model.predict_proba(scaled)[0][1] * 100 if model.n_classes_ == 2 else np.max(model.predict_proba(scaled)) * 100

    if prob < 33:
        risk = "Low Risk"
    elif prob < 66:
        risk = "Medium Risk"
    else:
        risk = "High Risk"

    print(f"Sample: {s}")
    print(f"→ Probability: {prob:.2f}%, Risk Level: {risk}\n")


✅ Model training complete!
✅ Model and Scaler saved as model.pkl and scaler.pkl
Sample: [45, 1, 95, 100, 5, 70, 168, 1, 0, 24.801587301587304]
→ Probability: 57.00%, Risk Level: Medium Risk

Sample: [30, 0, 98, 75, 2, 60, 160, 0, 0, 23.437499999999996]
→ Probability: 60.00%, Risk Level: Medium Risk

Sample: [60, 1, 88, 120, 9, 85, 170, 1, 1, 29.411764705882355]
→ Probability: 98.00%, Risk Level: High Risk

